# EdgeTwinCal paper figures
All quantitative values are loaded from `artifacts/edgetwincal_summary.csv`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

root = Path.cwd()
out = root / 'notebooks' / 'fig'
out.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':7,'pdf.fonttype':42})
fig, ax = plt.subplots(figsize=(7.1, 2.25))
ax.set_xlim(0, 15.5); ax.set_ylim(0, 4); ax.axis('off')
blue, teal, pale, gray = '#4C78A8', '#2CB1A1', '#D9F0ED', '#ECEFF3'
def box(x, y, w, h, text, fc, ec, lw=1.0):
    patch = FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.04,rounding_size=0.08',facecolor=fc,edgecolor=ec,linewidth=lw)
    ax.add_patch(patch); ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=7)
def arrow(x1,y1,x2,y2,style='-|>',color='#444444'):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle=style,mutation_scale=8,linewidth=1,color=color))
box(0.2,1.45,1.9,1.0,'Irregular sensor\nhistory',gray,'#7A7A7A')
box(2.65,1.2,2.45,1.5,'Frozen APN\nTAPA + query', '#E8F0F8',blue)
box(5.65,1.45,1.55,1.0,'Latent $h_n$', '#FFFFFF',blue)
box(7.75,2.65,2.15,0.85,'Frozen shared\ndecoder', '#E8F0F8',blue)
box(7.75,0.35,2.15,0.95,'SLRH\nlocal residual',pale,teal,1.4)
box(10.25,1.45,1.8,1.0,'Intermediate\nforecast', '#FFFFFF',teal)
box(12.45,1.2,1.65,1.5,'CFG\ncross-forecast\ngraph',pale,teal,1.4)
arrow(2.1,1.95,2.65,1.95); arrow(5.1,1.95,5.65,1.95)
arrow(7.2,2.05,7.75,2.95); arrow(7.2,1.85,7.75,0.82)
arrow(9.9,3.05,10.25,2.15); arrow(9.9,0.82,10.25,1.75)
arrow(12.05,1.95,12.45,1.95); arrow(14.1,1.95,14.45,1.95)
ax.text(14.55,1.95,'Final\nforecast',ha='left',va='center',fontsize=7,fontweight='bold',color=teal)
ax.text(3.88,0.72,'all APN parameters frozen',ha='center',va='center',fontsize=6.5,color=blue)
ax.text(10.8,0.18,'closed-form fitting; validation-selected shrinkage',ha='center',va='center',fontsize=6.5,color=teal)
fig.savefig(out/'Fig_1_overview.pdf',bbox_inches='tight')
fig.savefig(out/'Fig_1_overview.png',dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
import csv
import numpy as np

rows = list(csv.DictReader((root/'artifacts'/'edgetwincal_summary.csv').open()))
metrics = {
    metric: {row['variant']: row for row in rows if row['metric'] == metric}
    for metric in ('mse', 'mae')
}
seeds = (2024, 2025, 2026)
variants = ('slrh', 'cfg', 'full')
labels = ('SLRH', 'CFG', 'EdgeTwinCal')

def seed_values(metric, variant):
    return np.array([
        float(metrics[metric][variant][f'seed_{seed}']) for seed in seeds
    ])

gains = {}
for metric in ('mse', 'mae'):
    base = seed_values(metric, 'apn')
    gains[metric] = np.vstack([
        100 * (base - seed_values(metric, variant)) / base
        for variant in variants
    ])

fig, (ax_bar, ax_line) = plt.subplots(
    2, 1, figsize=(3.45, 4.35),
    gridspec_kw={'height_ratios': [1.0, 1.12]}
)

x = np.arange(len(variants))
width = 0.34
bar_colors = {'mse': '#4C78A8', 'mae': '#F58518'}
for offset, metric, label in [
    (-width/2, 'mse', 'MSE'),
    ( width/2, 'mae', 'MAE'),
]:
    means = gains[metric].mean(axis=1)
    stds = gains[metric].std(axis=1, ddof=1)
    bars = ax_bar.bar(
        x + offset, means, width, yerr=stds, capsize=2,
        color=bar_colors[metric], edgecolor='white', linewidth=0.5,
        label=label
    )
    for bar, mean in zip(bars, means):
        ax_bar.text(
            bar.get_x() + bar.get_width()/2, bar.get_height() + 0.045,
            f'{mean:.2f}', ha='center', va='bottom', fontsize=6
        )
ax_bar.axhline(0, color='#777777', linewidth=0.7)
ax_bar.set_ylabel('Improvement over APN (%)')
ax_bar.set_xticks(x, labels)
ax_bar.set_title('(a) Mean relative improvement', loc='left', fontsize=7)
ax_bar.legend(frameon=False, ncol=2, fontsize=6, loc='upper left')
ax_bar.spines[['top', 'right']].set_visible(False)
ax_bar.set_ylim(0, 1.32)

line_styles = {
    'apn': ('Frozen APN', '#555555', 'o'),
    'slrh': ('SLRH', '#54A24B', 's'),
    'cfg': ('CFG', '#8172B3', '^'),
    'full': ('EdgeTwinCal', '#2CB1A1', 'D'),
}
for variant, (label, color, marker) in line_styles.items():
    ax_line.plot(
        seeds, seed_values('mse', variant),
        label=label, color=color, marker=marker,
        linewidth=1.15, markersize=3.5
    )
ax_line.set_xlabel('Checkpoint seed')
ax_line.set_ylabel('Masked MSE')
ax_line.set_xticks(seeds)
ax_line.set_ylim(0.3080, 0.3135)
ax_line.set_title('(b) Seed-wise paired MSE', loc='left', fontsize=7)
ax_line.grid(axis='y', color='#E5E5E5', linewidth=0.6)
ax_line.legend(frameon=False, ncol=2, fontsize=5.7, loc='upper right')
ax_line.spines[['top', 'right']].set_visible(False)

fig.tight_layout(h_pad=1.25)
fig.savefig(out/'Fig_2_ablation.pdf', bbox_inches='tight')
fig.savefig(out/'Fig_2_ablation.png', dpi=300, bbox_inches='tight')
plt.show()
